# Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score


# Preparing Data

## Loading Data

In [59]:
train_data = pd.read_csv('train_clean.csv')
test_data = pd.read_csv('test_clean.csv')

## Preprocessing Data

In [64]:
X = train_data.drop(['PassengerId', "Name", "Cabin_Num", "Transported"], axis=1)
test_X = test_data.drop(['PassengerId', "Name", "Cabin_Num"], axis=1)
y = train_data['Transported']
y = y.astype(int)

In [65]:
X = pd.get_dummies(X)
test_X = pd.get_dummies(test_X)

In [66]:
X["CryoSleep"] = X["CryoSleep"].astype(int)
test_X["CryoSleep"] = test_X["CryoSleep"].astype(int)

X["VIP"] = X["VIP"].astype(int)
test_X["VIP"] = test_X["VIP"].astype(int)

In [67]:
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(
    [
        (
            "scaler",
            StandardScaler(),
            ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"],
        ),
    ],
    remainder="passthrough",
)
X_scaled = ct.fit_transform(X)
test_X_scaled = ct.transform(test_X)


In [68]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.1, random_state=42)

## Models

### Base Model

In [11]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [12]:
lr.score(X_test, y_test), lr.score(X_train, y_train)

(0.7701149425287356, 0.7938131151732072)

In [122]:
pred = lr.predict(test_X_scaled)

In [26]:
sample_sub = pd.read_csv('data/sample_submission.csv')

In [124]:
assert len(pred) == len(sample_sub)

In [125]:
sample_sub['Transported'] = pred

In [126]:
sample_sub.to_csv('base_submission.csv', index=False)

In [127]:
!kaggle competitions submit -c spaceship-titanic -f base_submission.csv -m "Base Model"

### XGB

In [ ]:
xgbc = XGBClassifier(max_depth=7, n_estimators=1000, learning_rate=0.05)
xgbc.fit(X_train, y_train)

XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
              importance_type=None, interaction_constraints='',
              learning_rate=0.05, max_bin=256, max_cat_to_onehot=4,
              max_delta_step=0, max_depth=7, max_leaves=0, min_child_weight=1,
              missing=nan, monotone_constraints='()', n_estimators=1000,
              n_jobs=0, num_parallel_tree=1, predictor='auto', random_state=0,
              reg_alpha=0, reg_lambda=1, ...)

`base_score=0.3, n_estimators=300, learning_rate=0.05, max_depth=5`

In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 7,],
    'n_estimators': [200, 300, 500],
    'learning_rate': [0.05, 0.1, 0.2],
    "base_score": [0.2, 0.3, 0.4],
}
xgb_base = XGBClassifier()
grid = GridSearchCV(xgb_base, param_grid, cv=3, scoring='accuracy', verbose=10)

grid.fit(X_scaled, y)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
[CV] base_score=0.2, learning_rate=0.05, max_depth=3, n_estimators=200 


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV]  base_score=0.2, learning_rate=0.05, max_depth=3, n_estimators=200, score=0.788, total=   1.5s
[CV] base_score=0.2, learning_rate=0.05, max_depth=3, n_estimators=200 


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    1.4s remaining:    0.0s


[CV]  base_score=0.2, learning_rate=0.05, max_depth=3, n_estimators=200, score=0.799, total=   0.7s
[CV] base_score=0.2, learning_rate=0.05, max_depth=3, n_estimators=200 


...

[CV]  base_score=0.2, learning_rate=0.05, max_depth=3, n_estimators=500, score=0.810, total=   1.7s
[CV] base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=200 


[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:   12.3s remaining:    0.0s


[CV]  base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=200, score=0.785, total=   1.6s
[CV] base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=200 
[CV]  base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=200, score=0.810, total=   0.9s
[CV] base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=200 
[CV]  base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=200, score=0.814, total=   1.0s
[CV] base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=300 
[CV]  base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=300, score=0.784, total=   2.2s
[CV] base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=300 
[CV]  base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=300, score=0.811, total=   1.5s
[CV] base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=300 
[CV]  base_score=0.2, learning_rate=0.05, max_depth=5, n_estimators=300, score=0.814, total=   1.2s
[CV] base_score=0.2, learning_rate=0.05,

[Parallel(n_jobs=1)]: Done 243 out of 243 | elapsed:  9.9min finished


GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     gamma=None, gpu_id=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None, max_bin=None,
                                     max_ca...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                  

In [16]:
grid.best_params_

{'base_score': 0.2, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 500}

In [18]:
xgb_best = grid.best_estimator_
xgb_best.fit(X_train, y_train)
xgb_best.score(X_test, y_test), xgb_best.score(X_train, y_train)

(0.7954022988505747, 0.8523584302697175)

In [19]:

param_grid = {
    'max_depth': [2, 3, 4],
    'n_estimators': [400, 500, 600],
    'learning_rate': [0.08, 0.1, 0.15],
    "base_score": [0.2, 0.25],
}
xgb_base = XGBClassifier()
grid = GridSearchCV(xgb_base, param_grid, cv=3, scoring='accuracy', verbose=10)

grid.fit(X_scaled, y)

Fitting 3 folds for each of 54 candidates, totalling 162 fits
[CV] base_score=0.2, learning_rate=0.08, max_depth=2, n_estimators=400 


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV]  base_score=0.2, learning_rate=0.08, max_depth=2, n_estimators=400, score=0.788, total=   2.3s
[CV] base_score=0.2, learning_rate=0.08, max_depth=2, n_estimators=400 


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.2s remaining:    0.0s


[CV]  base_score=0.2, learning_rate=0.08, max_depth=2, n_estimators=400, score=0.802, total=   1.5s
[CV] base_score=0.2, learning_rate=0.08, max_depth=2, n_estimators=400 


...

[CV]  base_score=0.2, learning_rate=0.08, max_depth=2, n_estimators=600, score=0.810, total=   2.1s
[CV] base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=400 


[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:   15.0s remaining:    0.0s


[CV]  base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=400, score=0.791, total=   1.9s
[CV] base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=400 
[CV]  base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=400, score=0.807, total=   2.0s
[CV] base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=400 
[CV]  base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=400, score=0.815, total=   1.3s
[CV] base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=500 
[CV]  base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=500, score=0.790, total=   1.6s
[CV] base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=500 
[CV]  base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=500, score=0.808, total=   3.5s
[CV] base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=500 
[CV]  base_score=0.2, learning_rate=0.08, max_depth=3, n_estimators=500, score=0.814, total=   5.8s
[CV] base_score=0.2, learning_rate=0.08,

[Parallel(n_jobs=1)]: Done 162 out of 162 | elapsed:  8.4min finished


GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     gamma=None, gpu_id=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None, max_bin=None,
                                     max_ca...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                  

In [21]:
grid.best_params_, grid.best_score_

({'base_score': 0.25,
  'learning_rate': 0.15,
  'max_depth': 3,
  'n_estimators': 500},
 0.8080072838968849)

In [22]:
xgb_best = grid.best_estimator_
xgb_best.fit(X_train, y_train)
xgb_best.score(X_test, y_test), xgb_best.score(X_train, y_train)

(0.7931034482758621, 0.8646299373641826)

In [23]:
pred = xgb_best.predict(test_X_scaled)
pred

array([1, 0, 1, ..., 1, 1, 1])

In [27]:
sample_sub["Transported"] = pred
sample_sub["Transported"] = sample_sub["Transported"].astype(bool)

In [28]:
sample_sub.to_csv("sub2.csv", index=False)

In [29]:
!kaggle competitions submit -c spaceship-titanic -f sub2.csv -m "XGB"

Successfully submitted to Spaceship Titanic


2022-06-19 11:58:05,752 WARNING Retrying (Retry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x000001734A1AE670>: Failed to establish a new connection: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond')': /api/v1/competitions/spaceship-titanic/submissions/url/61967/1655620056

  0%|          | 0.00/60.5k [00:00<?, ?B/s]
 13%|█▎        | 8.00k/60.5k [00:00<00:00, 64.5kB/s]
100%|██████████| 60.5k/60.5k [00:06<00:00, 10.1kB/s]


### SVC

In [ ]:
svc_clf = SVC(kernel='rbf', C=5, gamma=0.3)
svc_clf.fit(X_train, y_train)
svc_clf.score(X_test, y_test), svc_clf.score(X_train, y_train)

(0.7740080506037953, 0.8490077653149266)

In [ ]:
param_grid = {
    'C': [1, 3, 5, 10],
    'gamma': [0.01, 0.05, 0.1, 0.3, 0.5],
}
svc_base = SVC(kernel='rbf')
grid = GridSearchCV(svc_base, param_grid, cv=3, scoring='accuracy', verbose=10)
grid.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] C=1, gamma=0.01 .................................................


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV] ..................... C=1, gamma=0.01, score=0.774, total=   1.5s
[CV] C=1, gamma=0.01 .................................................


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    1.4s remaining:    0.0s


[CV] ..................... C=1, gamma=0.01, score=0.790, total=   1.2s
[CV] C=1, gamma=0.01 .................................................


...

[CV] ...................... C=1, gamma=0.1, score=0.802, total=   1.1s
[CV] C=1, gamma=0.3 ..................................................


[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:   10.6s remaining:    0.0s


[CV] ...................... C=1, gamma=0.3, score=0.777, total=   1.6s
[CV] C=1, gamma=0.3 ..................................................
[CV] ...................... C=1, gamma=0.3, score=0.784, total=   1.6s
[CV] C=1, gamma=0.3 ..................................................
[CV] ...................... C=1, gamma=0.3, score=0.794, total=   1.3s
[CV] C=1, gamma=0.5 ..................................................
[CV] ...................... C=1, gamma=0.5, score=0.771, total=   1.4s
[CV] C=1, gamma=0.5 ..................................................
[CV] ...................... C=1, gamma=0.5, score=0.778, total=   1.4s
[CV] C=1, gamma=0.5 ..................................................
[CV] ...................... C=1, gamma=0.5, score=0.794, total=   1.8s
[CV] C=3, gamma=0.01 .................................................
[CV] ..................... C=3, gamma=0.01, score=0.794, total=   1.0s
[CV] C=3, gamma=0.01 .................................................
[CV] .

[Parallel(n_jobs=1)]: Done  60 out of  60 | elapsed:  1.5min finished


GridSearchCV(cv=3, estimator=SVC(),
             param_grid={'C': [1, 3, 5, 10],
                         'gamma': [0.01, 0.05, 0.1, 0.3, 0.5]},
             scoring='accuracy', verbose=10)

In [ ]:
grid.best_params_

{'C': 10, 'gamma': 0.01}

In [ ]:
param_grid = {
    'C': [8, 10, 15, 20],
    'gamma': [0.005, 0.01, 0.02, 0.003],
}
svc_base = SVC(kernel='rbf')
grid = GridSearchCV(svc_base, param_grid, cv=3, scoring='accuracy', verbose=10)
grid.fit(X_train, y_train)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] C=8, gamma=0.005 ................................................


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV] .................... C=8, gamma=0.005, score=0.793, total=   1.6s
[CV] C=8, gamma=0.005 ................................................


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    1.5s remaining:    0.0s


[CV] .................... C=8, gamma=0.005, score=0.801, total=   1.1s
[CV] C=8, gamma=0.005 ................................................


...

[CV] ..................... C=8, gamma=0.02, score=0.817, total=   1.6s
[CV] C=8, gamma=0.003 ................................................


[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:   10.7s remaining:    0.0s


[CV] .................... C=8, gamma=0.003, score=0.785, total=   1.1s
[CV] C=8, gamma=0.003 ................................................
[CV] .................... C=8, gamma=0.003, score=0.798, total=   0.9s
[CV] C=8, gamma=0.003 ................................................
[CV] .................... C=8, gamma=0.003, score=0.806, total=   1.2s
[CV] C=10, gamma=0.005 ...............................................
[CV] ................... C=10, gamma=0.005, score=0.797, total=   1.0s
[CV] C=10, gamma=0.005 ...............................................
[CV] ................... C=10, gamma=0.005, score=0.801, total=   0.9s
[CV] C=10, gamma=0.005 ...............................................
[CV] ................... C=10, gamma=0.005, score=0.814, total=   1.0s
[CV] C=10, gamma=0.01 ................................................
[CV] .................... C=10, gamma=0.01, score=0.799, total=   0.9s
[CV] C=10, gamma=0.01 ................................................
[CV] .

[Parallel(n_jobs=1)]: Done  48 out of  48 | elapsed:   54.0s finished


GridSearchCV(cv=3, estimator=SVC(),
             param_grid={'C': [8, 10, 15, 20],
                         'gamma': [0.005, 0.01, 0.02, 0.003]},
             scoring='accuracy', verbose=10)

In [ ]:
grid.best_params_

{'C': 10, 'gamma': 0.01}

In [ ]:
svc_best = grid.best_estimator_.fit(X_train, y_train)
svc_best.score(X_test, y_test), svc_best.score(X_train, y_train)

(0.7889591719378953, 0.8155018694276676)

### Random Forest

In [ ]:
rfc = RandomForestClassifier(n_estimators=1000, max_depth=10, random_state=42)
rfc.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=1000, random_state=42)

In [ ]:
rfc.score(X_test, y_test), rfc.score(X_train, y_train)

(0.7958596894767107, 0.860943341961461)

In [ ]:
param_grid = {
    'n_estimators': [300, 500, 700, 900],
    'max_depth': [3, 5, 7, 9],
}
rfc_base = RandomForestClassifier()
grid = GridSearchCV(rfc_base, param_grid, cv=3, scoring='accuracy', verbose=10)
grid.fit(X_train, y_train)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] max_depth=3, n_estimators=300 ...................................


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV] ....... max_depth=3, n_estimators=300, score=0.736, total=   4.3s
[CV] max_depth=3, n_estimators=300 ...................................


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    4.2s remaining:    0.0s


[CV] ....... max_depth=3, n_estimators=300, score=0.739, total=   1.7s
[CV] max_depth=3, n_estimators=300 ...................................


...

[CV] ....... max_depth=3, n_estimators=700, score=0.751, total=   2.7s
[CV] max_depth=3, n_estimators=900 ...................................


[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:   22.3s remaining:    0.0s


[CV] ....... max_depth=3, n_estimators=900, score=0.733, total=   3.7s
[CV] max_depth=3, n_estimators=900 ...................................
[CV] ....... max_depth=3, n_estimators=900, score=0.742, total=   3.5s
[CV] max_depth=3, n_estimators=900 ...................................
[CV] ....... max_depth=3, n_estimators=900, score=0.752, total=   3.5s
[CV] max_depth=5, n_estimators=300 ...................................
[CV] ....... max_depth=5, n_estimators=300, score=0.767, total=   1.3s
[CV] max_depth=5, n_estimators=300 ...................................
[CV] ....... max_depth=5, n_estimators=300, score=0.773, total=   1.4s
[CV] max_depth=5, n_estimators=300 ...................................
[CV] ....... max_depth=5, n_estimators=300, score=0.784, total=   1.4s
[CV] max_depth=5, n_estimators=500 ...................................
[CV] ....... max_depth=5, n_estimators=500, score=0.764, total=   2.2s
[CV] max_depth=5, n_estimators=500 ...................................
[CV] .

[Parallel(n_jobs=1)]: Done  48 out of  48 | elapsed:  2.4min finished


GridSearchCV(cv=3, estimator=RandomForestClassifier(),
             param_grid={'max_depth': [3, 5, 7, 9],
                         'n_estimators': [300, 500, 700, 900]},
             scoring='accuracy', verbose=10)

In [ ]:
rfc_best = grid.best_estimator_.fit(X_train, y_train)
rfc_best.score(X_test, y_test), rfc_best.score(X_train, y_train)

(0.7941345600920069, 0.8449813057233247)

In [ ]:
grid.best_params_

{'max_depth': 9, 'n_estimators': 300}

In [ ]:
param_grid = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [8, 9, 10, 11],
}
rfc_base = RandomForestClassifier()
grid = GridSearchCV(rfc_base, param_grid, cv=3, scoring='accuracy', verbose=10)
grid.fit(X_train, y_train)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] max_depth=8, n_estimators=200 ...................................


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV] ....... max_depth=8, n_estimators=200, score=0.795, total=   1.9s
[CV] max_depth=8, n_estimators=200 ...................................


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    1.8s remaining:    0.0s


[CV] ....... max_depth=8, n_estimators=200, score=0.804, total=   1.1s
[CV] max_depth=8, n_estimators=200 ...................................


...

[CV] ....... max_depth=8, n_estimators=400, score=0.813, total=   2.0s
[CV] max_depth=8, n_estimators=500 ...................................


[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:   15.1s remaining:    0.0s


[CV] ....... max_depth=8, n_estimators=500, score=0.791, total=   2.7s
[CV] max_depth=8, n_estimators=500 ...................................
[CV] ....... max_depth=8, n_estimators=500, score=0.799, total=   2.5s
[CV] max_depth=8, n_estimators=500 ...................................
[CV] ....... max_depth=8, n_estimators=500, score=0.814, total=   2.7s
[CV] max_depth=9, n_estimators=200 ...................................
[CV] ....... max_depth=9, n_estimators=200, score=0.794, total=   1.1s
[CV] max_depth=9, n_estimators=200 ...................................
[CV] ....... max_depth=9, n_estimators=200, score=0.805, total=   1.1s
[CV] max_depth=9, n_estimators=200 ...................................
[CV] ....... max_depth=9, n_estimators=200, score=0.814, total=   1.2s
[CV] max_depth=9, n_estimators=300 ...................................
[CV] ....... max_depth=9, n_estimators=300, score=0.799, total=   1.6s
[CV] max_depth=9, n_estimators=300 ...................................
[CV] .

[Parallel(n_jobs=1)]: Done  48 out of  48 | elapsed:  1.6min finished


GridSearchCV(cv=3, estimator=RandomForestClassifier(),
             param_grid={'max_depth': [8, 9, 10, 11],
                         'n_estimators': [200, 300, 400, 500]},
             scoring='accuracy', verbose=10)

In [ ]:
grid.best_params_

{'max_depth': 10, 'n_estimators': 300}

In [ ]:
rfc_best = grid.best_estimator_.fit(X_train, y_train)
rfc_best.score(X_test, y_test), rfc_best.score(X_train, y_train)

(0.7975848188614146, 0.8596491228070176)

In [ ]:
pred = rfc_best.predict(scaled_text_X)
pred

array([ True, False,  True, ...,  True,  True,  True])

In [ ]:
sample_sub["Transported"] = pred
sample_sub.to_csv("sub3.csv", index=False)


In [ ]:
!kaggle competitions submit -c spaceship-titanic -f sub3.csv -m "Random Forest"

Successfully submitted to Spaceship Titanic



  0%|          | 0.00/60.5k [00:00<?, ?B/s]
 13%|█▎        | 8.00k/60.5k [00:00<00:00, 54.8kB/s]
100%|██████████| 60.5k/60.5k [00:05<00:00, 10.5kB/s]


### AdaBoost

In [ ]:
adc = AdaBoostClassifier(n_estimators=1000, learning_rate=0.2)
adc.fit(X_train, y_train)
adc.score(X_test, y_test), adc.score(X_train, y_train)

(0.7906843013225991, 0.8054357204486626)

In [ ]:
param_grid = {
    'n_estimators': [300, 500, 700, 900, 1000],
    'learning_rate': [0.05, 0.1, 0.2, 0.3],
}
adc_base = AdaBoostClassifier()
grid = GridSearchCV(adc_base, param_grid, cv=3, scoring='accuracy', verbose=10)
grid.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] learning_rate=0.05, n_estimators=300 ............................


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV]  learning_rate=0.05, n_estimators=300, score=0.786, total=   3.1s
[CV] learning_rate=0.05, n_estimators=300 ............................


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    3.0s remaining:    0.0s


[CV]  learning_rate=0.05, n_estimators=300, score=0.792, total=   2.8s
[CV] learning_rate=0.05, n_estimators=300 ............................


...

[CV]  learning_rate=0.05, n_estimators=700, score=0.805, total=   4.8s
[CV] learning_rate=0.05, n_estimators=900 ............................


[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:   34.4s remaining:    0.0s


[CV]  learning_rate=0.05, n_estimators=900, score=0.789, total=   6.5s
[CV] learning_rate=0.05, n_estimators=900 ............................
[CV]  learning_rate=0.05, n_estimators=900, score=0.796, total=   6.1s
[CV] learning_rate=0.05, n_estimators=900 ............................
[CV]  learning_rate=0.05, n_estimators=900, score=0.806, total=   7.2s
[CV] learning_rate=0.05, n_estimators=1000 ...........................
[CV]  learning_rate=0.05, n_estimators=1000, score=0.789, total=   7.1s
[CV] learning_rate=0.05, n_estimators=1000 ...........................
[CV]  learning_rate=0.05, n_estimators=1000, score=0.795, total=   6.9s
[CV] learning_rate=0.05, n_estimators=1000 ...........................
[CV]  learning_rate=0.05, n_estimators=1000, score=0.808, total=   7.8s
[CV] learning_rate=0.1, n_estimators=300 .............................
[CV] . learning_rate=0.1, n_estimators=300, score=0.788, total=   2.0s
[CV] learning_rate=0.1, n_estimators=300 .............................
[CV

[Parallel(n_jobs=1)]: Done  60 out of  60 | elapsed:  5.5min finished


GridSearchCV(cv=3, estimator=AdaBoostClassifier(),
             param_grid={'learning_rate': [0.05, 0.1, 0.2, 0.3],
                         'n_estimators': [300, 500, 700, 900, 1000]},
             scoring='accuracy', verbose=10)

In [ ]:
grid.best_params_

{'learning_rate': 0.1, 'n_estimators': 500}

In [ ]:
param_grid = {
    'n_estimators': [400, 500, 600],
    'learning_rate': [0.08, 0.1, 0.15],
}
adc_base = AdaBoostClassifier()
grid = GridSearchCV(adc_base, param_grid, cv=3, scoring='accuracy', verbose=10)
grid.fit(X_train, y_train)

Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] learning_rate=0.08, n_estimators=400 ............................


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV]  learning_rate=0.08, n_estimators=400, score=0.788, total=   9.8s
[CV] learning_rate=0.08, n_estimators=400 ............................


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    9.7s remaining:    0.0s


[CV]  learning_rate=0.08, n_estimators=400, score=0.795, total=   5.5s
[CV] learning_rate=0.08, n_estimators=400 ............................


...

[CV]  learning_rate=0.08, n_estimators=600, score=0.808, total=   4.0s
[CV] learning_rate=0.1, n_estimators=400 .............................


[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:   47.1s remaining:    0.0s


[CV] . learning_rate=0.1, n_estimators=400, score=0.789, total=   2.7s
[CV] learning_rate=0.1, n_estimators=400 .............................
[CV] . learning_rate=0.1, n_estimators=400, score=0.794, total=   2.8s
[CV] learning_rate=0.1, n_estimators=400 .............................
[CV] . learning_rate=0.1, n_estimators=400, score=0.805, total=   2.7s
[CV] learning_rate=0.1, n_estimators=500 .............................
[CV] . learning_rate=0.1, n_estimators=500, score=0.789, total=   3.6s
[CV] learning_rate=0.1, n_estimators=500 .............................
[CV] . learning_rate=0.1, n_estimators=500, score=0.796, total=   3.4s
[CV] learning_rate=0.1, n_estimators=500 .............................
[CV] . learning_rate=0.1, n_estimators=500, score=0.809, total=   3.4s
[CV] learning_rate=0.1, n_estimators=600 .............................
[CV] . learning_rate=0.1, n_estimators=600, score=0.789, total=   4.1s
[CV] learning_rate=0.1, n_estimators=600 .............................
[CV] .

[Parallel(n_jobs=1)]: Done  27 out of  27 | elapsed:  2.0min finished


GridSearchCV(cv=3, estimator=AdaBoostClassifier(),
             param_grid={'learning_rate': [0.08, 0.1, 0.15],
                         'n_estimators': [400, 500, 600]},
             scoring='accuracy', verbose=10)

In [ ]:
grid.best_params_, grid.best_score_

({'learning_rate': 0.1, 'n_estimators': 500}, 0.7979580097785447)

In [ ]:
ada_best = grid.best_estimator_.fit(X_train, y_train)
ada_best.score(X_test, y_test), ada_best.score(X_train, y_train)

(0.78953421506613, 0.8024158757549612)

In [ ]:
preds = ada_best.predict(scaled_text_X)
preds

array([ True, False,  True, ...,  True,  True,  True])

In [ ]:
sample_sub["Transported"] = preds
sample_sub.to_csv("sub4.csv", index=False)

In [ ]:
!kaggle competitions submit -c spaceship-titanic -f sub4.csv -m "Ada Boost"

Successfully submitted to Spaceship Titanic


  0%|          | 0.00/60.4k [00:00<?, ?B/s]
 13%|█▎        | 8.00k/60.4k [00:00<00:00, 67.5kB/s]
100%|██████████| 60.4k/60.4k [00:06<00:00, 8.93kB/s]


### Neural Network

In [76]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(
    [
        (
            "scaler",
            StandardScaler(),
            ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"],
        ),
        (
            "poly",
            PolynomialFeatures(degree=2),
            ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"],
        )
    ],
    remainder="passthrough",
)
X_scaled_n = ct.fit_transform(X)
test_X_scaled_n = ct.fit_transform(test_X)


In [77]:
X_scaled_n.shape

(8693, 52)

In [78]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_n, y, test_size=0.2, random_state=42
)

In [79]:
import tensorflow as tf
import tensorflow.keras.layers as tfl
from tensorflow.keras.models import Model

In [80]:
input = tfl.Input(shape=(X_train.shape[1],), name='input')
x = tfl.Dense(32, activation='relu')(input)
x = tfl.Dense(64, activation='relu')(x)
x = tfl.Dense(128, activation='relu')(x)
output = tfl.Dense(1, activation='sigmoid')(x)
model = Model(inputs=input, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input (InputLayer)          [(None, 52)]              0         
                                                                 
 dense_6 (Dense)             (None, 32)                1696      
                                                                 
 dense_7 (Dense)             (None, 64)                2112      
                                                                 
 dense_8 (Dense)             (None, 128)               8320      
                                                                 
 dense_9 (Dense)             (None, 1)                 129       
                                                                 
Total params: 12,257
Trainable params: 12,257
Non-trainable params: 0
_________________________________________________________________


In [81]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32,
          validation_data = (X_test, y_test))

Epoch 1/10
218/218 [==============================] - 35s 49ms/step - loss: 8942.0752 - accuracy: 0.7215 - val_loss: 7071.6064 - val_accuracy: 0.7355
Epoch 2/10
218/218 [==============================] - 12s 57ms/step - loss: 8841.3467 - accuracy: 0.7460 - val_loss: 8997.2188 - val_accuracy: 0.7688
Epoch 3/10
218/218 [==============================] - 12s 56ms/step - loss: 5065.6831 - accuracy: 0.7281 - val_loss: 10645.2285 - val_accuracy: 0.7706
Epoch 4/10
218/218 [==============================] - 6s 27ms/step - loss: 4252.5347 - accuracy: 0.7606 - val_loss: 4955.9316 - val_accuracy: 0.7608
Epoch 5/10
218/218 [==============================] - 4s 18ms/step - loss: 3266.2339 - accuracy: 0.7499 - val_loss: 3708.1628 - val_accuracy: 0.5595
Epoch 6/10
218/218 [==============================] - 4s 18ms/step - loss: 3143.0491 - accuracy: 0.7360 - val_loss: 4675.1074 - val_accuracy: 0.7188
Epoch 7/10
218/218 [==============================] - 2s 10ms/step - loss: 1803.3224 - accuracy: 0.732